<div class="alert alert-warning">

# PS 9 — The Rescorla–Wagner Model

In this problem set, you will implement and explore the **Rescorla–Wagner (RW) model** — one of the most influential computational models in the psychology of learning.

## Motivating scenario

You are training a dog. You ring a bell before every meal. After many repetitions the dog starts salivating at the sound of the bell alone. But what if you first trained the dog with a *light* until the light fully predicted the meal — and *only then* added the bell? The RW model makes a striking prediction: the dog will learn almost nothing about the bell, because the light already fully accounts for the food. This is the **blocking effect**, and it has been replicated extensively in both animals and humans.

## The model

The Rescorla–Wagner update rule is:
$$V_x(t+1) = V_x(t) + \beta\,\alpha_x\,\bigl[r - \textstyle\sum_y V_y(t)\bigr]$$

where:
- $V_x(t)$ — associative strength between CS $x$ and the US at trial $t$
- $r$ — reward on the current trial (1 = US present, 0 = absent)
- $\beta$ — global learning rate
- $\alpha_x$ — salience of CS $x$
- $\sum_y V_y(t)$ — total predicted reward, summed over **all** CSs present on this trial

The term $r - \sum_y V_y(t)$ is the **prediction error**: positive when the outcome is better than expected, negative when worse. The whole model is essentially one simple idea: *update your beliefs in proportion to how surprised you are.*

**Default parameters (unless otherwise stated):** $\alpha = 0.5$,  $\beta = 0.1$.

We will:
1. Simulate **acquisition** — learning that a CS predicts a reward
2. Demonstrate **blocking** — why a fully-predicted outcome blocks new learning
3. Simulate **extinction** — unlearning an association when reward stops
4. Explore **probabilistic reinforcement** — what the model learns when reward is uncertain

</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose

---

# 1. Acquisition

In a basic acquisition experiment, a single CS (e.g. a light) is repeatedly paired with a US (e.g. food). With no other CSs present, $\sum_y V_y = V_{\text{light}}$, so the update rule simplifies to:
$$V_{\text{light}}(t+1) = V_{\text{light}}(t) + \beta\,\alpha_{\text{light}}\,\bigl[r - V_{\text{light}}(t)\bigr]$$

This is an exponential approach toward $r$: the stronger the current association, the smaller the update.

<div class="alert alert-success">

## Exercise 1A&B

**A.** Implement `acquire(v0, T, beta, alpha, r)` that simulates $T$ acquisition trials and returns an array of length $T+1$ (including the initial value $V(0) =$ `v0`).

**B.** Plot 20 trials for two starting values: $V_{\text{light}}(0) = 0.05$ and $V_{\text{light}}(0) = 0.50$. Show both curves on the same plot as dotted lines with a legend. Label axes and include a title.


</div>

In [ ]:
# Default parameters
BETA  = 0.1   # global learning rate
ALPHA = 0.5   # CS salience


def acquire(v0, T=20, beta=BETA, alpha=ALPHA, r=1):
    """
    Simulate T acquisition trials of the RW model (single CS).

    Parameters
    ----------
    v0    : float  — initial association strength V(0)
    T     : int    — number of trials to simulate
    beta  : float  — global learning rate
    alpha : float  — CS salience
    r     : float  — reward value (1 = US present, 0 = absent)

    Returns
    -------
    V : ndarray of shape (T+1,)   — V(0), V(1), ..., V(T)
    """
    V = np.zeros(T + 1)
    V[0] = v0
    for t in range(T):
        # (TODO): Exercise 1A: replace pass with the update rule)
        pass
    return V

# ---- acquire tests ----

# Case 1: default params, start at 0
V = acquire(v0=0.0, T=5, beta=0.1, alpha=0.5, r=1.0)
assert_allclose(
    V,
    np.array([0.0, 0.05, 0.0975, 0.142625, 0.18549375, 0.2262190625]),
    rtol=1e-12, atol=1e-12,
)

# Case 2: same params, start at 0.05 (should be the same curve but shifted)
V = acquire(v0=0.05, T=5, beta=0.1, alpha=0.5, r=1.0)
assert_allclose(
    V,
    np.array([0.05, 0.0975, 0.142625, 0.18549375, 0.2262190625, 0.264908109375]),
    rtol=1e-12, atol=1e-12,
)

# Case 3: different learning rates (catches hard-coded defaults)
V = acquire(v0=0.0, T=5, beta=0.2, alpha=0.8, r=1.0)
assert_allclose(
    V,
    np.array([0.0, 0.16, 0.2944000000000001, 0.4072960000000001, 0.5021286400000001, 0.5817880576000001]),
    rtol=1e-12, atol=1e-12,
)

print('Success!')

In [ ]:
# (TODO): Exercise 1B: replace your code with the correct plot two initial conditions)
V_low  = 'YOUR_CODE_HERE'
V_high = 'YOUR_CODE_HERE'

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V_low,  'o--', label='$V(0) = 0.05$')
ax.plot(V_high, 's--', label='$V(0) = 0.50$')
ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V_{\mathrm{light}}$')
ax.set_title('Acquisition: effect of initial association strength')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 1C

**C.** Starting from $V_{\text{light}}(0) = 0.05$, how many trials does it take to *first exceed* $V_{\text{light}} = 0.8$? Find this numerically (e.g. with a `while` loop or `np.argmax`) and plot the trajectory for exactly that many trials.

</div>

In [ ]:
V = [0.05]

# (TODO) Exercise 1C: replace pass to find the number of trials to first exceed V = 0.8
while V[-1] <= 0.8:
    pass


n_trials = len(V) - 1
print(f'Trials to first exceed V = 0.8: {n_trials}')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V, 'o--', color='steelblue')
ax.axhline(0.8, color='gray', linestyle=':', label='$V = 0.8$')
ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V_{\mathrm{light}}$')
ax.set_title(f'Acquisition: {n_trials} trials to reach $V = 0.8$')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 1D (Optional challenge)

**D. (Optional challenge)** Suppose you are told it takes exactly 16 trials for $V$ to first exceed 0.4, starting from $V(0) = 0.0$. What is $\alpha_{\text{light}}$? Find the answer numerically and report it.

</div>

In [ ]:
# TODO Exercise 1D (optional): find alpha given that it takes 16 trials to first exceed V = 0.4


---

# 2. Blocking

The **blocking effect** (Kamin, 1969) is one of the RW model's most famous predictions. If a CS (the light) has already been fully conditioned, adding a second CS (the bell) to the compound produces almost no new learning for the bell — because the light already fully predicts the reward and the prediction error is near zero.

With two CSs present on the same trial, both update based on the *same* shared prediction error:
$$V_{\text{light}}(t+1) = V_{\text{light}}(t) + \beta\,\alpha_{\text{light}}\,\bigl[r - V_{\text{light}}(t) - V_{\text{bell}}(t)\bigr]$$
$$V_{\text{bell}}(t+1)  = V_{\text{bell}}(t)  + \beta\,\alpha_{\text{bell}} \,\bigl[r - V_{\text{light}}(t) - V_{\text{bell}}(t)\bigr]$$

<div class="alert alert-success">

## Exercise 2A&B

**A.** Implement `acquire_two_cs(v0_light, v0_bell, T, beta, alpha_light, alpha_bell, r)` that simulates compound conditioning with two CSs and returns both association-strength arrays (each of length $T+1$).

**B.** Simulate 30 trials with $V_{\text{light}}(0) = 0.8$, $V_{\text{bell}}(0) = 0.0$, and $\alpha_{\text{bell}} = 0.2$ (keep $\alpha_{\text{light}} = 0.5$, $\beta = 0.1$). Plot $V_{\text{bell}}$ as a function of trial number.

</div>

In [ ]:
ALPHA_BELL = 0.2

def acquire_two_cs(v0_light, v0_bell, T=30, beta=BETA,
                   alpha_light=ALPHA, alpha_bell=ALPHA_BELL, r=1):
    """
    Simulate compound conditioning with two CSs.

    Returns
    -------
    V_light, V_bell : ndarrays of shape (T+1,)
    """
    V_light = np.zeros(T + 1)
    V_bell  = np.zeros(T + 1)
    V_light[0] = v0_light
    V_bell[0]  = v0_bell
    # (TODO) Exercise 2A: replace pass to update V_light and V_bell
    for t in range(T):
        pass
    
    return V_light, V_bell


# ---- acquire_two_cs tests ----
# Case 1: both start at zero, unequal salience
V_light, V_bell = acquire_two_cs(
    v0_light=0.0, v0_bell=0.0,
    T=5, beta=0.1, alpha_light=0.5, alpha_bell=0.2, r=1.0
)
assert_allclose(
    V_light,
    np.array([0.0, 0.05, 0.0965, 0.139745, 0.17996285, 0.2173654505]),
    rtol=1e-12, atol=1e-12,
)
assert_allclose(
    V_bell,
    np.array([0.0, 0.02, 0.03860000000000001, 0.05589800000000002, 0.07198514000000002, 0.08694618020000001]),
    rtol=1e-12, atol=1e-12,
)

# Case 2: equal salience symmetry check (should match exactly)
V_light, V_bell = acquire_two_cs(
    v0_light=0.0, v0_bell=0.0,
    T=5, beta=0.1, alpha_light=0.5, alpha_bell=0.5, r=1.0
)
assert_allclose(
    V_light,
    np.array([0.0, 0.05, 0.095, 0.1355, 0.17195, 0.204755]),
    rtol=1e-12, atol=1e-12,
)
assert_allclose(
    V_bell,
    np.array([0.0, 0.05, 0.095, 0.1355, 0.17195, 0.204755]),
    rtol=1e-12, atol=1e-12,
)


print('Success!')    

In [ ]:
# (TODO) Exercise 2B: replace YOUR_CODE_HERE to simulate and plot blocking
V_light, V_bell ='YOUR_CODE_HERE'

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V_bell,  'o--', color='steelblue',  label='$V_{\mathrm{bell}}$')
ax.plot(V_light, 's--', color='darkorange', label='$V_{\mathrm{light}}$')
ax.set_xlabel('Trial')
ax.set_ylabel('Association strength')
ax.set_title('Blocking: pre-trained light prevents bell from learning')
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 2C

**C.** In 1–2 sentences: why does the RW model predict that the bell learns so little? What would happen to $V_{\text{bell}}$ if we started with $V_{\text{light}}(0) = 0.0$ instead?

</div>

**2C. Your Answer:** 

---

# 3. Extinction

Once a CS has acquired a strong association, it can be **extinguished** by repeatedly presenting the CS *without* the US ($r = 0$). The prediction error becomes negative on every trial and $V$ decays back toward 0. This models the lab finding that a previously conditioned response diminishes when the US is withheld.

<div class="alert alert-success">

## Exercise 3A

**A.** First run 20 **acquisition** trials starting from $V(0) = 0.0$ (use `acquire` with default parameters). Then run 20 **extinction** trials — call `acquire` again with the same $\alpha$ and $\beta$ but $r = 0$, starting from where acquisition ended. Concatenate the two phases and plot $V$ across all 40 trials. Mark the acquisition–extinction transition with a vertical dashed line and label both phases.

</div>

In [ ]:
# (TODO) Exercise 3A: replace 'YOUR_CODE_HERE' for acquisition followed by extinction
V_acq = 'YOUR_CODE_HERE'
V_ext = 'YOUR_CODE_HERE'

# Concatenate, dropping the duplicated transition point
V_all = np.concatenate([V_acq, V_ext[1:]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(V_all, 'o--', color='steelblue')
# (TODO) Mark the acquisition–extinction transition with a vertical dashed line using ax.axvline (one line)

# (TODO) Label both the acquisition and extinction phases using ax.text (two lines)

ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V$')
ax.set_title('Acquisition followed by extinction')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3B

**B.** Does extinction speed depend on $\alpha$? Repeat the same acquisition + extinction procedure for $\alpha \in \{0.2,\, 0.5,\, 0.8\}$ (keeping $\beta = 0.1$, $V(0) = 0$). Plot all three trajectories on one figure with a legend. In 1–2 sentences, explain what you observe.

</div>

In [ ]:
# Exercise 3B: extinction rate for different values of alpha
alphas = [0.2, 0.5, 0.8]
colors = ['steelblue', 'darkorange', 'tomato']

fig, ax = plt.subplots(figsize=(8, 4))
for alpha, color in zip(alphas, colors):
    # (TODO) replace 'YOUR_CODE_HERE' for extinction rate for different values of alpha
    V_acq = 'YOUR_CODE_HERE'
    V_ext = 'YOUR_CODE_HERE'
    V_all = np.concatenate([V_acq, V_ext[1:]])
    ax.plot(V_all, 'o--', color=color, label=f'$\\alpha = {alpha}$')

# (TODO) Mark the acquisition–extinction transition with a vertical dashed line using ax.axvline (one line)

# (TODO) Label both the acquisition and extinction phases using ax.text (two lines)

ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V$')
ax.set_title('Effect of salience on acquisition and extinction rate')
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.show()

**3B. Your Answer:** 

---

# 4. Probabilistic Reinforcement

So far, reward has been delivered on every trial. In the real world, reinforcement is rarely perfectly reliable. What does the RW model predict when reward occurs with some probability $P$?

<div class="alert alert-success">

## Exercise 4A

**A. Alternating reinforcement.** Suppose trials alternate: bell + food, then bell alone, then bell + food, and so on ($r$ alternates 1, 0, 1, 0, …). Starting from $V_{\text{bell}}(0) = 0$ with $\alpha_{\text{bell}} = 0.5$, $\beta = 0.1$, simulate 50 trials and plot the trajectory. To what value does $V_{\text{bell}}$ appear to converge? Provide an intuitive explanation in 1–2 sentences.


</div>

In [ ]:
# Exercise 4A: alternating reinforcement
ALPHA_BELL = .2
BETA = .5
T = 50
V = np.zeros(T + 1)
# (TODO) replace 'YOUR_CODE_HERE' for alternating reinforcement
outcomes = ['YOUR_CODE_HERE']   # 1, 0, 1, 0, …

for t, r in enumerate(outcomes):
    # (TODO) replace 'pass' for V update
    pass

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(V, 'o--', color='steelblue')
ax.axhline(0.5, color='tomato', linestyle=':', label='$V = 0.5$')
ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V_{\mathrm{bell}}$')
ax.set_title('Alternating reinforcement')
ax.legend()
plt.tight_layout()
plt.show()

**4A. Your Answer:** 

<div class="alert alert-success">

## Exercise 4B

**B. Partial reinforcement.** Now suppose food is delivered independently on each trial with probability $P$ (draw $r \sim \text{Bernoulli}(P)$ on each trial). For each of $P \in \{0.2,\, 0.6,\, 0.8\}$:
- Run **20 independent simulations** of 100 trials each (different random seeds).
- Plot the **mean trajectory** with a **shaded 95% confidence band** (± 1.96 × standard error across the 20 runs).

Arrange the three plots as **3 rows × 1 column** with an overall figure size of `(7, 15)`. Include $P$ in each subplot title, and add shared axis labels.

*Hint:* Use `np.random.default_rng(seed)` and `rng.random() < P` to generate a Bernoulli sample for each trial.

</div>

In [ ]:
# Exercise 4B: partial reinforcement — 20 independent simulations per P, shaded confidence band
T      = 100
n_sims = 20
probs  = [0.2, 0.6, 0.8]
trials = np.arange(T + 1)

fig, axes = plt.subplots(3, 1, figsize=(7, 15), sharex=True)
fig.suptitle('Partial reinforcement', fontsize=14)

for ax, P in zip(axes, probs):
    runs = np.zeros((n_sims, T + 1))
    for sim in range(n_sims):
        # (TODO) replace 'YOUR_CODE_HERE' to initialize a Bernoulli sampler
        rng = 'YOUR_CODE_HERE'
        V = np.zeros(T + 1)
        for t in range(T):
            # (TODO) replace 'pass' to draw a Bernoulli sample and update V
            pass
        
        runs[sim] = V
    
    # (TODO) replace 'YOUR_CODE_HERE' to compute the mean and standard error across runs
    mean = 'YOUR_CODE_HERE'
    se   = 'YOUR_CODE_HERE'

    ax.plot(trials, mean, color='steelblue')
    ax.fill_between(trials, mean - 1.96 * se, mean + 1.96 * se,
                    alpha=0.3, color='steelblue')
    ax.axhline(P, color='tomato', linestyle='--', linewidth=1.5, label=f'$P = {P}$')
    ax.set_title(f'$P = {P}$')
    ax.set_ylabel('$V_{\\mathrm{bell}}$')
    ax.set_ylim([-0.05, 1.05])
    ax.legend(loc='lower right')

axes[-1].set_xlabel('Trial')
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 4C

**C.** To what value does the association converge (approximately) for each $P$? How does this compare to $P$ itself? What does this tell you about what the RW model is learning under partial reinforcement?

</div>

**4C. Your Answer:** 

---

# 5. Overshadowing

Blocking is a special case of a more general phenomenon: whenever two CSs are conditioned **together**, they must share the available prediction error. Even starting from $V = 0$, the two CSs compete — each ends up with a weaker association than it would have acquired if trained alone. This is called **overshadowing**.

With two equally salient CSs ($\alpha_1 = \alpha_2 = 0.5$), the total association $V_1 + V_2$ still converges to $r = 1$, but each CS only reaches $V \approx 0.5$. With unequal saliences, the more salient CS "takes" more of the total association, leaving less for the weaker one.

<div class="alert alert-success">

## Exercise 5A

**A.** Using `acquire_two_cs`, simulate 30 trials of compound conditioning starting from $V_{\text{CS1}}(0) = V_{\text{CS2}}(0) = 0$, with equal saliences $\alpha_{\text{CS1}} = \alpha_{\text{CS2}} = 0.5$. On the same figure, also plot single-CS acquisition with $\alpha = 0.5$ for comparison. In 1–2 sentences, describe what you observe.

</div>

In [ ]:
# Exercise 5A: equal-salience overshadowing vs. single-CS acquisition
# (TODO) replace 'YOUR_CODE_HERE' for equal-salience overshadowing
V_cs1, V_cs2 = 'YOUR_CODE_HERE'
# (TODO) replace 'YOUR_CODE_HERE' for single-CS acquisition                               
V_single = 'YOUR_CODE_HERE'

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(V_single, 'k--',          label='Single CS ($\\alpha = 0.5$)')
ax.plot(V_cs1,    'o-',  color='steelblue',  label='CS1 in compound ($\\alpha = 0.5$)')
ax.plot(V_cs2,    's--', color='darkorange', label='CS2 in compound ($\\alpha = 0.5$)')
ax.set_xlabel('Trial')
ax.set_ylabel('Association strength $V$')
ax.set_title('Overshadowing: equal-salience compound vs. single CS')
ax.legend()
plt.tight_layout()
plt.show()

**5A. Your Answer:** 

<div class="alert alert-success">

## Exercise 5B

**B.** Repeat with unequal saliences: $\alpha_{\text{CS1}} = 0.5$ and $\alpha_{\text{CS2}} = 0.1$. Plot both $V_{\text{CS1}}$ and $V_{\text{CS2}}$. Which CS is more overshadowed, and why?

</div>

In [ ]:
# (TODO) Exercise 5B: unequal-salience overshadowing
V_strong, V_weak = 'YOUR_CODE_HERE'

# (TODO) plot V_strong, V_weak similar to the exercise 5A plot
fig, ax = plt.subplots(figsize=(7, 4))

plt.tight_layout()
plt.show()

**5B. Your Answer:** 

---

# 6. Latent Inhibition

In lab experiments, animals that are **pre-exposed** to a CS alone (without any US) subsequently learn the CS–US association more slowly than animals with no prior exposure. This is called **latent inhibition**, and it has been found across many species and conditioning paradigms.

What does the RW model predict?

<div class="alert alert-success">

## Exercise 6A

**A.** Simulate two groups, each followed by 20 acquisition trials ($r = 1$, default parameters):

- **Pre-exposed group:** 20 pre-exposure trials with the CS alone ($r = 0$), starting from $V(0) = 0$; then 20 acquisition trials starting from wherever pre-exposure ended.
- **Naive group:** 20 acquisition trials only, starting from $V(0) = 0$.

Plot both full trajectories on the same figure (marking the onset of acquisition for the pre-exposed group with a vertical dashed line). Does the RW model predict slower learning for the pre-exposed group?

</div>

In [ ]:
# (TODO) Exercise 6A: pre-exposed group vs. naive group
V_preexp_phase1 = 'YOUR_CODE_HERE'   # pre-exposure: CS alone, no US
V_preexp_phase2 = 'YOUR_CODE_HERE'   # acquisition
V_preexp_all    = np.concatenate([V_preexp_phase1, V_preexp_phase2[1:]])

V_naive = 'YOUR_CODE_HERE'   # naive: straight into acquisition

fig, ax = plt.subplots(figsize=(8, 4))
# (TODO) use V_preexp_all and V_naive to plot the two trajectories

ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 6B

**B.** In 2–3 sentences: why does the RW model fail to predict latent inhibition? What would a model need in order to capture this effect?

</div>

**6B. Your Answer:** 

---

<div class="alert alert-warning">

When you are done:

1. **Restart the kernel and re-run the whole notebook** (`Kernel → Restart & Run All`) to make sure everything runs without errors.
2. Make sure you can answer all questions and understand all the code and outputs.

</div>